# Week 4 · Notebook 04: The Twist. How Good Is *Your* Pipeline?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/04_injection_recovery.ipynb)

You found a planet. Great. Now suppose you search 1,000 stars and find 10 planets. **How many did you miss?**
Without that number you can say nothing about how common planets are: "we found few small planets" might mean
small planets are rare, or that your pipeline is blind to them.

The answer is an experiment: inject fake planets of *known* size and period into the data, run your pipeline
**blind**, and count how many come back. The fraction recovered is your **detection efficiency**
(*completeness*). This is exactly how the Kepler team calibrated its final catalogue. **Time:** the whole week.

### 🎯 By the end of this notebook you can
1. inject a fake transit correctly (into raw data, before detrending);
2. define and defend a recovery criterion;
3. run an injection–recovery grid and read a completeness map, with error bars;
4. summarise your pipeline with one number, $S_{50}$;
5. measure the false-alarm rate, the other side of choosing a threshold;
6. design and run a controlled experiment on your own pipeline.

### 🧭 Roadmap
| § | Topic |
|---|---|
| 1 | One injection, by hand |
| 2 | What counts as "recovered"? |
| 3 | A handful of injections in a loop |
| 4 | The full grid: completeness map and efficiency curve |
| 5 | False alarms: the other half of a threshold |
| 6 | Real data |
| 7 | Your experiment |

**Reading:**
- Christiansen et al. 2013 ([ADS](https://ui.adsabs.harvard.edu/abs/2013ApJS..207...35C)), the first Kepler injection–recovery
- Christiansen et al. 2020, DR25 completeness ([arXiv:2010.04796](https://arxiv.org/abs/2010.04796)), Sec. 1–3
- Burke et al. 2015 ([ADS](https://ui.adsabs.harvard.edu/abs/2015ApJ...809....8B)): how completeness turns into occurrence rates (skim)
<details><summary><b>📎 How to read this notebook (click to open)</b></summary>

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | Before running the next cell, write down what you expect. Being wrong is how you learn. |
| 👀 **What to look for** | How to read the output or plot you just made. |
| ✏️ **Your turn** | An exercise. 🟢 everyone · 🟡 stretch · 🔴 challenge. |
| 💡 / ✅ | Click to reveal a hint or a worked solution. Try first, then peek. |
| ⚠️ **Pitfall** | A mistake that real researchers make. |
| 🧠 **Concept check** | Short quiz. Answers are hidden under each question. |
| `check(...)` | Self-check: prints ✅ when your exercise answer is right. |

Run cells **top to bottom** with **Shift+Enter**. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).
</details>

In [ ]:
# ⚙️ SETUP: run this cell first, every time you open the notebook. Works in Google Colab and locally.
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})

def check(name, value, expected, rtol=0.02):
    """Self-check for the exercises. Prints ✅, ❌, or ⏳ (not attempted yet)."""
    if value is None:
        print(f"⏳ {name}: not attempted yet (replace None with your answer)")
        return
    try:
        ok = bool(np.allclose(value, expected, rtol=rtol))
    except Exception:
        ok = value == expected
    print(f"✅ {name}: correct!" if ok else f"❌ {name}: not quite (you gave {value!r}). Check the hint.")

print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, inject, plotting, utils
from transitpipe.io import LightCurve
import pandas as pd

## 1. One injection, by hand
**Rule #1: inject into the RAW light curve, before detrending.** Detrending is one of the main ways a pipeline
loses planets (Notebook 02 §4). Injecting *after* detrending would hide that loss and make your pipeline look
better than it is.

We start on a synthetic star (fast, offline, and we know there are no real planets in it), then move to real data in §6.

In [ ]:
raw = inject.make_synthetic_lightcurve(seed=5)         # a spotted star, 27 days, 30-min cadence, 300 ppm noise
fake, truth = inject.inject_transit(raw, period=6.0, rp_rearth=3.0, seed=1)
print({k: round(float(v), 4) for k, v in truth.items()})

PIPELINE = dict(method="median", window_days=1.0)      # 🔒 freeze YOUR pipeline settings here
flat = detrend.sigma_clip(detrend.flatten(fake, **PIPELINE))
r = search.bls(flat)
print(r.summary())

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(fake.time, fake.flux, ".", ms=1, color="k")
for k in range(5):
    ax[0].axvline(truth["t0"] + k * truth["period"], color="C3", alpha=0.4)
ax[0].set_xlabel("Time [d]"); ax[0].set_ylabel("Flux"); ax[0].set_title("Raw + injected (red = true transit times)")
plotting.plot_periodogram(r, ax=ax[1], true_period=truth["period"])
plotting.plot_fold(flat, r.period, r.t0, ax=ax[2], window_hours=8)
plt.tight_layout(); plt.show()

🔮 **Predict:** change `rp_rearth` to **2.0**, then **1.5**, then **1.0**, and rerun. At what size does the
periodogram stop pointing at 6 d?

👀 At some size the true peak sinks into the forest of noise peaks and BLS returns a wrong period: the planet is
lost. But one injection tells you nothing about *how often* that happens; noise and phase are random. For that
you need many injections.

## 2. What counts as "recovered"?
You must decide, and write it in your report. Ours (in `inject._one_trial`):
1. found period within **1 %** of the injected one (aliases like 2:1 are logged but *not* counted);
2. found mid-time within one transit duration of an injected transit (right period, wrong dip doesn't count);
3. **SDE ≥ 7**.

Change any of these and your completeness changes. That is not a bug, it is the point: completeness is a property
of the *whole* pipeline, criteria included.

In [ ]:
for found in [6.03, 6.2, 3.0, 12.0, 4.0, 9.0]:
    match, kind = search.is_match(found, 6.0)
    print(f"found {found:5.2f} d vs injected 6.00 d -> match = {match!s:5}, kind = {kind}")

## 3. A handful of injections in a loop
Before the full grid, run a few by hand so you know exactly what the package does. Same period, shrinking radius,
a different random phase each time.

🔮 **Predict:** sketch "recovered yes/no vs radius". Will it switch sharply or gradually?

In [ ]:
noise_ppm = detrend.estimate_noise_ppm(detrend.flatten(raw, **PIPELINE))
rows = []
for i, rp in enumerate([4.0, 3.0, 2.5, 2.0, 1.7, 1.5, 1.3, 1.1, 1.0, 0.8]):
    inj, tr = inject.inject_transit(raw, period=6.0, rp_rearth=rp, seed=10 + i)
    rr = search.bls(detrend.sigma_clip(detrend.flatten(inj, **PIPELINE)), oversample=1.0)
    match, kind = search.is_match(rr.period, tr["period"])
    rows.append({"rp_inj": rp, "snr_expected": inject.expected_snr(noise_ppm, tr["depth"], tr["n_in_transit"]),
                 "period_found": rr.period, "sde": rr.sde, "kind": kind, "recovered": match and rr.sde >= 7})
small = pd.DataFrame(rows)
small.round(2)

👀 Recovery tracks the **expected SNR** much better than the radius alone, and the switch from found to
missed is not a sharp line: near the threshold, luck (where the transits fall, how the noise happens to behave)
decides. That's why we need many injections per cell and report a *fraction*.

## 4. The full grid
Now the real experiment: periods and radii drawn **log-uniformly** inside each cell of a grid, several injections per
cell, the frozen pipeline run blind on each. Running in parallel (`n_jobs`) uses all your CPU cores.

The output is a table (pandas DataFrame) with one row per injection:

| column | meaning |
|---|---|
| `period_inj`, `rp_inj`, `t0_inj`, `depth_inj` | what we injected (the truth) |
| `snr_expected` | $(\delta/\sigma)\sqrt{N_\text{in}}$ for that planet |
| `period_found`, `sde`, `snr` | what BLS returned |
| `match_kind` | `exact`, an alias like `2:1`, `wrong_phase`, `below_threshold`, or `none` |
| `recovered` | True / False by the criteria in §2 |

In [ ]:
period_edges = np.geomspace(0.7, 12, 6)      # 5 period bins, log-spaced
radius_edges = np.geomspace(0.7, 6, 6)       # 5 radius bins
df = inject.injection_recovery(raw, period_edges, radius_edges, n_per_cell=8,
                               detrend_kwargs=PIPELINE, bls_kwargs={"oversample": 1.0},
                               n_jobs=os.cpu_count(), progress=False)
print(len(df), "injections, overall recovered fraction:", round(df.recovered.mean(), 2))
df.head()

In [ ]:
inject.plot_completeness(df, period_edges, radius_edges); plt.show()

👀 **Read the map.** Big planets and short periods: bright (found). Small and long: dark (missed).
Why does completeness fall with period? Fewer transits in 27 days → fewer in-transit points → lower SNR.
Longer transits are also partly eaten by the detrending window.

### How sure are we about each cell?
Each cell has only a few injections, so each fraction is uncertain. For $k$ recovered out of $n$, the
**binomial** error is $\sigma_f = \sqrt{f(1-f)/n}$. With $n = 8$ and $f = 0.5$ that's ±0.18: big! More injections
per cell shrink it as $1/\sqrt{n}$.

In [ ]:
frac, n = inject.completeness_map(df, period_edges, radius_edges)
err = np.sqrt(frac * (1 - frac) / n)
labels_P = [f"{a:.1f}–{b:.1f} d" for a, b in zip(period_edges[:-1], period_edges[1:])]
labels_R = [f"{a:.1f}–{b:.1f} R⊕" for a, b in zip(radius_edges[:-1], radius_edges[1:])]
cells = pd.DataFrame([[f"{f:.2f} ± {e:.2f}" for f, e in zip(fr, er)] for fr, er in zip(frac.T, err.T)],
                     index=labels_R, columns=labels_P)
cells.iloc[::-1]          # biggest radii on top, like the map

### One number for your pipeline: $S_{50}$
Period and radius both act through one variable, the **expected SNR**. Plot recovery against it and fit an S-curve
(a logistic function); the SNR where it crosses 50 % is $S_{50}$, a single honest summary of your pipeline.

In [ ]:
plotting.plot_efficiency(df); plt.show()
s50, width, _ = inject.fit_efficiency_curve(df)
print(f"Your pipeline finds half of all planets with expected SNR = {s50:.1f} (transition width {width:.1f})")

In [ ]:
df.match_kind.value_counts()     # how did the failures fail?

👀 `none` = lost in the noise; `2:1` / `1:2` = found an alias; `below_threshold` = right period but SDE < 7;
`wrong_phase` = right period, wrong dip. Each failure mode suggests a different fix. Which dominates for you?

## 5. False alarms: the other half of a threshold
Lowering the SDE threshold finds more planets, but also "finds" planets that aren't there. To measure the
**false-alarm rate**, run the frozen pipeline on light curves with **no planet** and record the best SDE each time.

🔮 **Predict:** what fraction of planet-free light curves will exceed SDE 7?

In [ ]:
max_sde = []
for s in range(20):
    empty = inject.make_synthetic_lightcurve(seed=100 + s)        # same kind of star, different noise, NO planet
    rr = search.bls(detrend.sigma_clip(detrend.flatten(empty, **PIPELINE)), oversample=1.0)
    max_sde.append(rr.sde)
max_sde = np.array(max_sde)

plt.figure(figsize=(6, 3.5))
plt.hist(max_sde, bins=12, alpha=0.7, label="no planet injected")
plt.hist(df.sde[df.recovered], bins=20, alpha=0.5, label="recovered injections")
plt.axvline(7, color="C3", ls="--", label="threshold SDE = 7")
plt.xlabel("best SDE"); plt.ylabel("count"); plt.legend(); plt.show()
right_signal = df.recovered | (df.match_kind == "below_threshold")   # right period and phase, any SDE
for thr in [6, 7, 8, 9]:
    print(f"threshold {thr}: false-alarm fraction {np.mean(max_sde >= thr):.2f}, "
          f"completeness {np.mean(right_signal & (df.sde >= thr)):.2f}")

👀 Surprised? On this spotted star, the *best noise peak* reaches SDE ≈ 7 in most planet-free light curves. The
textbook threshold doesn't transfer to every star and pipeline. That's why you must measure it. (Our recovery
criteria survive this because they also demand the right period *and* phase, which noise almost never matches.)

Every threshold is a trade-off between **completeness** (finding real planets) and **reliability** (not
reporting fake ones). Kepler's final catalogue measured both: injections for completeness, and inverted /
scrambled light curves for false alarms (Thompson et al. 2018). With only 20 noise-only curves, your false-alarm
fraction is itself uncertain; a full-timer can run hundreds.

## 6. Now on real data
Real noise isn't white Gaussian: there are residual systematics, gaps, and the real planet. **Remove the known
planet first**, or your pipeline keeps finding Kepler-10 b instead of the fake one. (Try it without masking once:
it's an instructive failure.)

In [ ]:
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3])
lc.meta.update(rstar=1.065, mstar=0.910)
REAL_PIPE = dict(method="median", window_days=0.5)
# Get the real planet's ephemeris from YOUR OWN search (published ephemerides drift over the years)
r0 = search.bls(detrend.flatten(lc, **REAL_PIPE), max_period=10)
known = detrend.transit_mask(lc.time, r0.period, r0.t0, r0.duration, pad=2.0)
lc_clean = LightCurve(lc.time[~known], lc.flux[~known], lc.flux_err[~known], lc.meta)
print(f"known planet at P = {r0.period:.5f} d; removed {known.sum()} in-transit points")

pe = np.geomspace(0.6, 20, 5); re = np.geomspace(0.5, 4, 5)
df_real = inject.injection_recovery(lc_clean, pe, re, n_per_cell=6, detrend_kwargs=REAL_PIPE,
                                    bls_kwargs={"oversample": 1.0, "max_period": 25},
                                    n_jobs=os.cpu_count(), progress=False)
inject.plot_completeness(df_real, pe, re, title="Kepler-10, Q3: detection efficiency"); plt.show()
print("overall:", round(df_real.recovered.mean(), 2))
try:
    print(f"S50 on real data: {inject.fit_efficiency_curve(df_real)[0]:.1f}  (synthetic: {s50:.1f})")
except Exception as err:
    print("efficiency fit failed (too few injections near the threshold):", err)

👀 Kepler-10 is bright and quiet, so even ~1 R⊕ planets at short periods are often recovered: far better
than our noisy synthetic star. Compare $S_{50}$ between the two. If real data need a higher SNR for the same
recovery, the extra losses come from real-world noise that the simple SNR formula ignores.

## 7. The science question: what limits *your* pipeline?
This is where your project becomes *yours*. Pick **one** question, run a **controlled experiment** (change one
thing at a time, same random seed, same grid), and answer it with completeness maps and $S_{50}$ values.

1. **Detrending window:** run the grid for windows 0.25, 0.5, 1 and 2 d. Long-duration (long-period) planets
   should suffer at short windows. Do they? Where's the sweet spot?
2. **Detrending method:** median vs savgol vs biweight. Does Hippke et al.'s conclusion hold for your star?
3. **Detection threshold:** SDE 6 vs 7 vs 9, with false alarms from §5. Which threshold would you choose, and why?
4. **BLS grid:** `oversample` 0.5 vs 1 vs 3. Does a coarse period grid miss planets? Which ones?
5. **Kepler vs TESS:** same planet sizes, a Kepler quarter vs a TESS sector of a similar-brightness star.
6. **Quiet vs active star:** pick a spotted, fast-rotating star. How much completeness do you lose?

⚠️ **Pitfall:** don't tune your pipeline on the same injections you report. Choose settings, *freeze* them, then run
a fresh grid with a new seed for the final numbers.

<details><summary>💡 Template for experiment 1</summary>

```python
results = {}
for w in [0.25, 0.5, 1.0, 2.0]:
    results[w] = inject.injection_recovery(lc_clean, pe, re, n_per_cell=6,
                      detrend_kwargs=dict(method="median", window_days=w),
                      bls_kwargs={"oversample": 1.0, "max_period": 25}, seed=0,
                      n_jobs=os.cpu_count(), progress=False)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (w, d) in zip(axes, results.items()):
    inject.plot_completeness(d, pe, re, ax=ax, title=f"window {w} d")
plt.tight_layout(); plt.show()
for w, d in results.items():
    print(w, round(inject.fit_efficiency_curve(d)[0], 1))
```
</details>

In [ ]:
# ✏️ YOUR EXPERIMENT HERE

### 🚀 Going further (full-timers)
- Scale up: 20+ injections per cell and a finer grid, on a lab machine. Do your error bars shrink as $1/\sqrt{n}$?
- Fit a **gamma CDF** instead of a logistic (Christiansen et al. use this) and compare.
- Turn completeness into an **occurrence rate**: with $N_\text{found}$ planets in a cell from $N_\star$ stars,
  $f_\text{occ} \approx N_\text{found} / (N_\star \cdot \text{completeness} \cdot p_\text{transit})$, where
  $p_\text{transit}\approx R_\star/a$ is the chance the orbit is aligned to transit at all.
- Inject limb-darkened transits (`model="batman"` in `inject.transit_signal`). Does completeness change?

## 🧠 Concept check
**Q1.** Why inject before detrending, not after?
<details><summary>Answer</summary>Detrending can distort or erase transits. Injecting after it would hide those losses and overestimate completeness.</details>

**Q2.** A cell has 3 of 12 injections recovered. What is the completeness and its error?
<details><summary>Answer</summary>f = 0.25, σ = √(0.25·0.75/12) ≈ 0.13.</details>

**Q3.** You lower the threshold from SDE 9 to 6. What happens to completeness and false alarms?
<details><summary>Answer</summary>Both go up. You find more real planets and more fake ones; the right threshold depends on how many false alarms you can afford to vet.</details>

**Q4.** You found 2 planets with 1–2 R⊕ and P = 5–10 d among 500 stars; completeness there is 0.4 and transit
probability ~0.05. Roughly what fraction of stars host such a planet?
<details><summary>Answer</summary>2 / (500 × 0.4 × 0.05) = 0.2, so about 20 % (with a large error from only 2 planets!).</details>

## 📌 Key takeaways
- Completeness = fraction of injected planets your *frozen* pipeline recovers blind.
- Inject into raw data; define recovery criteria explicitly; freeze settings before the final run.
- Recovery depends mainly on expected SNR; $S_{50}$ summarises a pipeline in one number.
- Every threshold trades completeness against false alarms. Measure both.
- Occurrence rate = found / (stars × completeness × transit probability).

## 📖 Glossary
**injection–recovery**: inserting fake signals and checking whether they are found · **completeness / detection
efficiency**: fraction recovered · **reliability**: fraction of detections that are real · **false alarm**: a
detection caused by noise · **$S_{50}$**: SNR at 50 % recovery · **occurrence rate**: fraction of stars hosting a
given kind of planet.

### ✅ Week 4 Checkpoint (final)
- [ ] Synthetic + real completeness maps committed
- [ ] Efficiency curve + your $S_{50}$ number
- [ ] False-alarm test and your chosen threshold, justified
- [ ] One controlled experiment from §7 with a clear conclusion
- [ ] Report (4–6 pages) + 5-minute talk / poster: see the project guide, Section 6

🎉 **You built a planet-finding pipeline and measured its limits, the same way professional surveys do.**